# vLLM Serving Runner (Colab GPU)

Run this notebook on a GPU runtime to launch a vLLM OpenAI-compatible endpoint and export a public URL for local benchmarking.

In [ ]:
import os, re, sys, time, subprocess
from urllib import request, error

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'vllm', 'pyngrok'])
print('Installed: vllm, pyngrok')

In [ ]:
MODEL_NAME = os.environ.get('VLLM_MODEL', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0')
PORT = int(os.environ.get('VLLM_PORT', '8000'))
LOG_PATH = '/tmp/vllm_server.log'
os.environ['VLLM_ATTENTION_BACKEND'] = os.environ.get('VLLM_ATTENTION_BACKEND', 'XFORMERS')

if 'vllm_proc' in globals() and vllm_proc is not None and vllm_proc.poll() is None:
    vllm_proc.terminate(); time.sleep(2)
if 'vllm_cfd' in globals() and vllm_cfd is not None and vllm_cfd.poll() is None:
    vllm_cfd.terminate(); time.sleep(1)

server_cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--host', '0.0.0.0', '--port', str(PORT), '--model', MODEL_NAME,
    '--dtype', 'float16', '--gpu-memory-utilization', '0.40', '--max-model-len', '512', '--enforce-eager'
]
print('Starting:', ' '.join(server_cmd))
vllm_proc = subprocess.Popen(server_cmd, stdout=open(LOG_PATH, 'w'), stderr=subprocess.STDOUT, text=True)
time.sleep(25)

PUBLIC_BASE_URL = None
try:
    from pyngrok import ngrok
    if os.environ.get('NGROK_AUTHTOKEN'): ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])
    PUBLIC_BASE_URL = ngrok.connect(PORT, 'http').public_url.rstrip('/')
    print('Using ngrok tunnel')
except Exception as exc:
    print('ngrok unavailable, fallback to cloudflared:', exc)

if PUBLIC_BASE_URL is None:
    subprocess.run(['bash', '-lc', "which cloudflared || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared)"], check=True)
    vllm_cfd = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    pat = re.compile(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com')
    t0 = time.time()
    while time.time() - t0 < 35:
        line = vllm_cfd.stdout.readline() if vllm_cfd.stdout else ''
        if not line: time.sleep(0.2); continue
        m = pat.search(line)
        if m: PUBLIC_BASE_URL = m.group(0); break

print('PUBLIC_BASE_URL=', PUBLIC_BASE_URL)

for _ in range(30):
    try:
        with request.urlopen(request.Request(PUBLIC_BASE_URL + '/v1/models', method='GET'), timeout=5) as resp:
            print('Health HTTP', resp.status)
            print(resp.read(300).decode('utf-8', 'replace'))
            break
    except Exception as exc:
        print('Health retry:', type(exc).__name__, exc)
        time.sleep(2)

print('vLLM running:', vllm_proc.poll() is None)
print('Use PUBLIC_BASE_URL in local config generation.')

In [ ]:
print('Local machine commands:')
print(f"uv run python scripts/cloud/write_remote_config.py --backend vllm --base-url {PUBLIC_BASE_URL} --r5 --output configs/live_vllm_remote_r5.json")
print('uv run python scripts/probe_live_endpoints.py --config configs/live_vllm_remote_r5.json')
print('uv run mws-bench sweep --config configs/live_vllm_remote_r5.json --output results/live_vllm_remote_sweep_r5.csv --trace-output-dir results/traces/live_vllm_remote_r5')